# Performance Bugs Extraction and Analysis

This notebook implements the data extraction methodology from Section IV of the paper:
"Fixing Performance Bugs Through LLM Explanations"

## Goals:
1. Extract bugs from 17 Defects4J projects
2. Filter for performance-related bugs (target: 490 bugs)
3. Analyze bug characteristics and distribution
4. Prepare dataset for categorization

In [ ]:
import sys
import os
sys.path.append('..')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List
import logging

# Configure visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# Import our modules
from config import DEFECTS4J_PROJECTS, DATA_DIR
from extraction.enhanced_extractor import EnhancedDefects4JExtractor
from utils.java_parser import PerformancePatternDetector

## 1. Extract Bugs from Defects4J

Following the paper's methodology, we extract bugs from all 17 projects.

In [ ]:
# Initialize extractor
extractor = EnhancedDefects4JExtractor()

# Check if we have cached results
cache_file = DATA_DIR / "all_bugs_extracted.json"

if cache_file.exists():
    print("Loading cached bugs...")
    with open(cache_file, 'r') as f:
        all_bugs_data = json.load(f)
    print(f"Loaded {len(all_bugs_data)} bugs from cache")
else:
    print("Extracting bugs from Defects4J projects...")
    print("This will take several hours on first run.")
    
    # Extract from each project
    all_bugs = []
    for project in DEFECTS4J_PROJECTS[:3]:  # Start with first 3 for testing
        print(f"\nExtracting from {project}...")
        bugs = extractor.extract_project_bugs(project)
        all_bugs.extend(bugs)
        print(f"  Extracted {len(bugs)} bugs")
    
    # Save to cache
    all_bugs_data = [bug.to_dict() for bug in all_bugs]
    with open(cache_file, 'w') as f:
        json.dump(all_bugs_data, f, indent=2)
    print(f"\nTotal bugs extracted: {len(all_bugs)}")

In [ ]:
# Convert to DataFrame for analysis
df_bugs = pd.DataFrame(all_bugs_data)
print(f"Total bugs: {len(df_bugs)}")
print(f"\nColumns: {df_bugs.columns.tolist()}")
df_bugs.head()

## 2. Filter Performance Bugs

Apply performance filtering criteria from Section IV.A of the paper.

In [ ]:
# Filter for performance bugs
df_perf = df_bugs[df_bugs['is_performance_bug'] == True].copy()
print(f"Performance bugs: {len(df_perf)} ({len(df_perf)/len(df_bugs)*100:.1f}%)")

# Show performance score distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df_bugs['performance_score'], bins=20, alpha=0.7, edgecolor='black')
plt.axvline(x=0.3, color='r', linestyle='--', label='Threshold')
plt.xlabel('Performance Score')
plt.ylabel('Count')
plt.title('Performance Score Distribution')
plt.legend()

plt.subplot(1, 2, 2)
df_perf.groupby('project').size().plot(kind='bar')
plt.xlabel('Project')
plt.ylabel('Number of Performance Bugs')
plt.title('Performance Bugs by Project')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 3. Analyze Performance Keywords

Examine which keywords are most indicative of performance bugs.

In [ ]:
# Analyze keyword frequency
from collections import Counter

all_keywords = []
for keywords in df_perf['performance_keywords']:
    if isinstance(keywords, list):
        all_keywords.extend(keywords)

keyword_counts = Counter(all_keywords)

# Plot top keywords
top_keywords = dict(keyword_counts.most_common(15))

plt.figure(figsize=(12, 5))
plt.bar(range(len(top_keywords)), list(top_keywords.values()))
plt.xticks(range(len(top_keywords)), list(top_keywords.keys()), rotation=45, ha='right')
plt.xlabel('Keyword')
plt.ylabel('Frequency')
plt.title('Top Performance Keywords in Commit Messages')
plt.tight_layout()
plt.show()

print("Most common performance indicators:")
for keyword, count in keyword_counts.most_common(10):
    print(f"  {keyword}: {count}")

## 4. Analyze Code Changes

Examine the nature of performance fixes.

In [ ]:
# Analyze code change metrics
df_perf['lines_changed'] = df_perf['added_lines'] + df_perf['removed_lines']

# Statistics
print("Code Change Statistics:")
print(f"  Average lines added: {df_perf['added_lines'].mean():.1f}")
print(f"  Average lines removed: {df_perf['removed_lines'].mean():.1f}")
print(f"  Average total change: {df_perf['lines_changed'].mean():.1f}")
print(f"  Median total change: {df_perf['lines_changed'].median():.1f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df_perf['lines_changed'], bins=30, edgecolor='black')
axes[0].set_xlabel('Lines Changed')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Code Change Size')

axes[1].scatter(df_perf['added_lines'], df_perf['removed_lines'], alpha=0.5)
axes[1].set_xlabel('Lines Added')
axes[1].set_ylabel('Lines Removed')
axes[1].set_title('Added vs Removed Lines')

# Files modified per bug
files_per_bug = df_perf['modified_files'].apply(len)
axes[2].hist(files_per_bug, bins=20, edgecolor='black')
axes[2].set_xlabel('Number of Files Modified')
axes[2].set_ylabel('Count')
axes[2].set_title('Files Modified per Bug')

plt.tight_layout()
plt.show()

## 5. Extract Method-Level Changes

Analyze performance improvements at the method level.

In [ ]:
# Analyze method-level changes
method_changes = []
for _, bug in df_perf.iterrows():
    if isinstance(bug['modified_methods'], list):
        for method in bug['modified_methods']:
            if isinstance(method, dict):
                method_changes.append({
                    'bug_id': bug['identifier'],
                    'method_name': method.get('method_name', 'unknown'),
                    'buggy_complexity': method.get('buggy_complexity', 0),
                    'fixed_complexity': method.get('fixed_complexity', 0),
                    'complexity_reduction': method.get('buggy_complexity', 0) - method.get('fixed_complexity', 0)
                })

df_methods = pd.DataFrame(method_changes)

if len(df_methods) > 0:
    print(f"Total method changes: {len(df_methods)}")
    print(f"Methods with complexity reduction: {(df_methods['complexity_reduction'] > 0).sum()}")
    print(f"Average complexity reduction: {df_methods['complexity_reduction'].mean():.2f}")
    
    # Show complexity changes
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    plt.hist(df_methods['complexity_reduction'], bins=20, edgecolor='black')
    plt.xlabel('Complexity Reduction')
    plt.ylabel('Count')
    plt.title('Distribution of Complexity Reduction')
    
    plt.subplot(1, 2, 2)
    plt.scatter(df_methods['buggy_complexity'], df_methods['fixed_complexity'], alpha=0.5)
    plt.plot([0, 20], [0, 20], 'r--', label='No change')
    plt.xlabel('Buggy Complexity')
    plt.ylabel('Fixed Complexity')
    plt.title('Complexity Before vs After Fix')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

## 6. Pattern Analysis

Identify common performance bug patterns.

In [ ]:
# Analyze patterns in buggy code
pattern_detector = PerformancePatternDetector()

pattern_counts = {
    'algorithmic_inefficiency': 0,
    'memory_usage': 0,
    'redundant_computation': 0,
    'cpu_overhead': 0,
    'io_inefficiency': 0
}

# Sample analysis on first 50 bugs
for _, bug in df_perf.head(50).iterrows():
    if isinstance(bug['modified_methods'], list):
        for method in bug['modified_methods']:
            if isinstance(method, dict) and 'buggy_code' in method:
                patterns = pattern_detector.detect_patterns(method['buggy_code'])
                for category in patterns:
                    pattern_counts[category] += 1

# Visualize pattern distribution
if sum(pattern_counts.values()) > 0:
    plt.figure(figsize=(10, 5))
    categories = list(pattern_counts.keys())
    counts = list(pattern_counts.values())
    
    plt.bar(categories, counts, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57'])
    plt.xlabel('Pattern Category')
    plt.ylabel('Count')
    plt.title('Performance Bug Patterns Detected (Sample)')
    plt.xticks(rotation=45, ha='right')
    
    # Add percentage labels
    total = sum(counts)
    for i, (cat, count) in enumerate(zip(categories, counts)):
        if total > 0:
            plt.text(i, count + 0.5, f'{count/total*100:.1f}%', ha='center')
    
    plt.tight_layout()
    plt.show()
    
    print("Pattern distribution:")
    for cat, count in pattern_counts.items():
        print(f"  {cat}: {count}")

## 7. Prepare Dataset for Next Steps

Create the filtered dataset of 490 performance bugs as described in the paper.

In [ ]:
# Select top 490 performance bugs based on score
df_perf_sorted = df_perf.sort_values('performance_score', ascending=False)
df_final = df_perf_sorted.head(490)

print(f"Final dataset: {len(df_final)} performance bugs")
print(f"\nProject distribution:")
print(df_final.groupby('project').size().sort_values(ascending=False))

# Save final dataset
output_file = DATA_DIR / "performance_bugs_490.json"
df_final.to_json(output_file, orient='records', indent=2)
print(f"\nSaved to {output_file}")

## 8. Summary Statistics

Generate summary statistics matching the paper's description.

In [ ]:
print("="*50)
print("PERFORMANCE BUGS DATASET SUMMARY")
print("="*50)
print(f"\nTotal bugs extracted: {len(df_bugs)}")
print(f"Performance bugs identified: {len(df_perf)}")
print(f"Final dataset size: {len(df_final)}")
print(f"\nPerformance bug rate: {len(df_perf)/len(df_bugs)*100:.1f}%")
print(f"\nProjects covered: {df_final['project'].nunique()}")
print(f"Average bugs per project: {len(df_final)/df_final['project'].nunique():.1f}")
print(f"\nCode change statistics:")
print(f"  Average lines changed: {df_final['lines_changed'].mean():.1f}")
print(f"  Median lines changed: {df_final['lines_changed'].median():.1f}")
print(f"  Average files modified: {df_final['modified_files'].apply(len).mean():.1f}")
print(f"\nTop 5 performance keywords:")
for keyword, count in keyword_counts.most_common(5):
    print(f"  - {keyword}: {count}")